# Task 3 — Biological and Clinical Interpretation

Analyze **real TCGA-BRCA data** to identify biologically relevant features and perform statistical tests.

**Analysis:**
- Identify top variable genes/proteins from TCGA breast cancer data
- Statistical tests (t-test) for Basal-like (responders) vs Other subtypes
- Feature importance from ML models
- Mutation-phenotype associations for known breast cancer driver genes

## 1. Initialize Project Environment

In [1]:
"""Setup and imports for biological interpretation."""
import logging
import sys
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

print(f"Python {sys.version}")
print(f"numpy {np.__version__}")
print(f"pandas {pd.__version__}")

Python 3.12.3 (main, Jan  8 2026, 11:30:50) [GCC 13.3.0]
numpy 2.1.3
pandas 2.2.3


## 2. Define Configuration Parameters

In [2]:
@dataclass
class Task3Config:
    """Configuration for biological interpretation."""

    handle: str = "AndreiCod"
    export_dir: Path = Path("./artifacts")
    data_dir: Path = Path("./artifacts")
    random_seed: int = 42
    significance_threshold: float = 0.05
    top_features_to_analyze: int = 20

    def __post_init__(self):
        self.export_dir.mkdir(parents=True, exist_ok=True)

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["export_dir"] = str(info["export_dir"])
        info["data_dir"] = str(info["data_dir"])
        return info


CONFIG = Task3Config()
CONFIG.describe()

{'handle': 'AndreiCod',
 'export_dir': 'artifacts',
 'data_dir': 'artifacts',
 'random_seed': 42,
 'significance_threshold': 0.05,
 'top_features_to_analyze': 20}

## 3. Load TCGA-BRCA Data from Previous Tasks

In [3]:
# Load TCGA-BRCA preprocessed data
expr_df = pd.read_csv(CONFIG.data_dir / "task1_expression_data.csv", index_col=0)
prot_df = pd.read_csv(CONFIG.data_dir / "task1_proteomics_data.csv", index_col=0)
snp_df = pd.read_csv(CONFIG.data_dir / "task1_snp_data.csv", index_col=0)
pheno_df = pd.read_csv(CONFIG.data_dir / "task1_phenotypes.csv", index_col=0)

# Load integrated features from Task 2
X_integrated = pd.read_csv(
    CONFIG.data_dir / "task2_integrated_features.csv", index_col=0
)

# Prepare phenotype
y = pheno_df["phenotype"].map({"responder": 1, "non_responder": 0})

print(f"{'=' * 60}")
print("LOADED TCGA-BRCA DATA")
print(f"{'=' * 60}")
print(f"Expression: {expr_df.shape}")
print(f"Proteomics: {prot_df.shape}")
print(f"SNP/Mutations: {snp_df.shape}")
print(f"Phenotypes: {pheno_df.shape}")
print(f"Integrated features: {X_integrated.shape}")
print(f"\nMolecular Subtypes:")
print(pheno_df["molecular_subtype"].value_counts())

LOADED TCGA-BRCA DATA
Expression: (1000, 277)
Proteomics: (208, 277)
SNP/Mutations: (277, 100)
Phenotypes: (277, 8)
Integrated features: (277, 400)

Molecular Subtypes:
molecular_subtype
BRCA_LumA      122
BRCA_LumB       62
BRCA_Basal      55
BRCA_Her2       22
BRCA_Normal      9
Name: count, dtype: int64


## 4. Identify Top Variable Features

In [4]:
def get_top_variable_features(
    df: pd.DataFrame, n_top: int, feature_type: str
) -> pd.DataFrame:
    """Get top variable features with their statistics."""
    variances = df.var(axis=1)
    means = df.mean(axis=1)
    stds = df.std(axis=1)

    stats_df = pd.DataFrame(
        {
            "feature": df.index,
            "type": feature_type,
            "variance": variances,
            "mean": means,
            "std": stds,
            "cv": stds / (means + 1e-10),  # Coefficient of variation
        }
    ).sort_values("variance", ascending=False)

    return stats_df.head(n_top)


# Get top variable genes and proteins
top_genes = get_top_variable_features(expr_df, CONFIG.top_features_to_analyze, "gene")
top_proteins = get_top_variable_features(
    prot_df, CONFIG.top_features_to_analyze, "protein"
)

print("Top Variable Genes:")
print(top_genes[["feature", "variance", "mean"]].head(10).to_string(index=False))
print("\nTop Variable Proteins:")
print(top_proteins[["feature", "variance", "mean"]].head(10).to_string(index=False))

Top Variable Genes:
 feature  variance     mean
    4250 27.369284 8.726264
   10143 26.300019 4.696756
    1360 25.726593 6.581383
   10647 21.168944 6.688904
    7031 20.916825 8.049366
    5304 19.023947 8.097154
  118430 18.266636 6.774652
    1556 18.165101 7.385006
  199974 16.509695 6.144065
    6278 16.481078 3.378034

Top Variable Proteins:
 feature  variance          mean
     142  1.090909  2.220446e-16
     836  1.090909 -7.031412e-16
     842  1.090909 -2.960595e-16
   10987  1.090909 -8.326673e-16
    8289  1.003968 -1.053176e-17
    1027  1.003623 -2.693393e-16
    2597  1.003623  3.847704e-17
   25937  1.003623 -5.130272e-16
    6714  1.003623 -1.288981e-15
   -3967  1.003623 -5.386786e-16


## 5. Biological Relevance Analysis

### Known Cancer-Related Features

In [5]:
# Known breast cancer-related genes from TCGA-BRCA data
# These are the top mutated genes in our dataset (from cBioPortal)
CANCER_GENE_ROLES = {
    "PIK3CA": "PI3K/AKT pathway oncogene, mutated in 36% of breast cancers",
    "TP53": "Tumor suppressor, 'guardian of the genome', frequently mutated in Basal-like",
    "CDH1": "E-cadherin, tumor suppressor, lobular carcinoma marker",
    "GATA3": "Transcription factor, luminal differentiation, frequently mutated",
    "MAP3K1": "MAPK pathway, mutated in luminal breast cancers",
    "TTN": "Titin, large gene often mutated (may be passenger)",
    "MUC16": "CA-125, often mutated in cancers, may be passenger",
    "KMT2C": "Histone methyltransferase, chromatin remodeling",
    "SYNE1": "Nuclear envelope protein, structural role",
    "RYR2": "Calcium channel, unclear cancer relevance",
}

# RPPA protein names from TCGA-BRCA
CANCER_PROTEIN_ROLES = {
    "TP53": "p53 tumor suppressor protein",
    "EGFR": "Epidermal growth factor receptor, therapy target",
    "ERBB2": "HER2, oncogenic receptor, breast cancer therapy target",
    "ESR1": "Estrogen receptor alpha, hormone therapy target",
    "PGR": "Progesterone receptor, hormone receptor status marker",
    "BRCA1": "DNA repair complex component",
    "PIK3CA": "PI3K catalytic subunit, PI3K/AKT pathway",
    "AKT": "AKT kinase, PI3K/AKT pathway",
    "PTEN": "Tumor suppressor, PI3K pathway inhibitor",
    "BCL2": "Anti-apoptotic protein, survival marker",
}


def annotate_cancer_relevance(
    features: pd.DataFrame, annotations: Dict
) -> pd.DataFrame:
    """Add cancer relevance annotations to features."""
    features = features.copy()
    features["cancer_role"] = features["feature"].map(annotations).fillna("Unknown")
    features["is_known_cancer_gene"] = features["feature"].isin(annotations.keys())
    return features


top_genes_annotated = annotate_cancer_relevance(top_genes, CANCER_GENE_ROLES)
top_proteins_annotated = annotate_cancer_relevance(top_proteins, CANCER_PROTEIN_ROLES)

# Show known cancer genes in top variable features
known_cancer_genes = top_genes_annotated[top_genes_annotated["is_known_cancer_gene"]]
print(f"Known cancer genes in top {CONFIG.top_features_to_analyze} variable genes:")
if len(known_cancer_genes) > 0:
    for _, row in known_cancer_genes.iterrows():
        print(f"  - {row['feature']}: {row['cancer_role']}")
else:
    print("  (Note: Gene names are Entrez IDs, lookup required for annotation)")

Known cancer genes in top 20 variable genes:
  (Note: Gene names are Entrez IDs, lookup required for annotation)


## 6. Statistical Tests: Responders vs Non-Responders

In [6]:
def differential_expression_analysis(
    expr_df: pd.DataFrame, pheno: pd.Series, significance_threshold: float = 0.05
) -> pd.DataFrame:
    """
    Perform t-test to identify differentially expressed genes.

    Compares responders vs non-responders.
    """
    responder_samples = pheno[pheno == "responder"].index
    non_responder_samples = pheno[pheno == "non_responder"].index

    results = []

    for gene in expr_df.index:
        resp_vals = expr_df.loc[gene, responder_samples].values
        non_resp_vals = expr_df.loc[gene, non_responder_samples].values

        # Perform t-test
        t_stat, p_value = stats.ttest_ind(resp_vals, non_resp_vals)

        # Calculate fold change (log2)
        mean_resp = resp_vals.mean()
        mean_non_resp = non_resp_vals.mean()
        log2fc = mean_resp - mean_non_resp  # Already log2 scale

        results.append(
            {
                "gene": gene,
                "mean_responder": mean_resp,
                "mean_non_responder": mean_non_resp,
                "log2_fold_change": log2fc,
                "t_statistic": t_stat,
                "p_value": p_value,
            }
        )

    results_df = pd.DataFrame(results)

    # Multiple testing correction (Benjamini-Hochberg)
    from scipy.stats import rankdata

    n = len(results_df)
    ranked_pvals = rankdata(results_df["p_value"])
    results_df["p_adjusted"] = results_df["p_value"] * n / ranked_pvals
    results_df["p_adjusted"] = results_df["p_adjusted"].clip(upper=1.0)

    results_df["significant"] = results_df["p_adjusted"] < significance_threshold
    results_df = results_df.sort_values("p_value")

    return results_df


de_results = differential_expression_analysis(
    expr_df, pheno_df["phenotype"], CONFIG.significance_threshold
)

sig_genes = de_results[de_results["significant"]]
print(
    f"Differentially expressed genes (p_adj < {CONFIG.significance_threshold}): {len(sig_genes)}"
)
print("\nTop significant genes:")
print(
    de_results.head(10)[
        ["gene", "log2_fold_change", "p_value", "p_adjusted", "significant"]
    ].to_string(index=False)
)

Differentially expressed genes (p_adj < 0.05): 690

Top significant genes:
 gene  log2_fold_change      p_value   p_adjusted  significant
 3169         -6.378726 3.383431e-90 3.383431e-87         True
10551         -7.673470 2.611173e-73 1.305586e-70         True
 7494         -3.242141 1.307603e-72 4.358676e-70         True
79083         -4.898214 2.930742e-69 7.326856e-67         True
25803         -5.186003 1.055586e-62 2.111171e-60         True
 7033         -7.422347 1.549170e-59 2.581951e-57         True
 2625         -4.523612 3.936233e-58 5.623190e-56         True
 2099         -6.612971 6.860381e-55 8.575477e-53         True
23158         -3.879497 8.935424e-55 9.928249e-53         True
  771         -4.626441 6.174715e-51 6.174715e-49         True


In [7]:
def differential_protein_analysis(
    prot_df: pd.DataFrame, pheno: pd.Series, significance_threshold: float = 0.05
) -> pd.DataFrame:
    """Perform t-test on protein abundance."""
    responder_samples = pheno[pheno == "responder"].index
    non_responder_samples = pheno[pheno == "non_responder"].index

    results = []

    for protein in prot_df.index:
        resp_vals = prot_df.loc[protein, responder_samples].values
        non_resp_vals = prot_df.loc[protein, non_responder_samples].values

        t_stat, p_value = stats.ttest_ind(resp_vals, non_resp_vals)

        mean_diff = resp_vals.mean() - non_resp_vals.mean()

        results.append(
            {
                "protein": protein,
                "mean_responder": resp_vals.mean(),
                "mean_non_responder": non_resp_vals.mean(),
                "mean_difference": mean_diff,
                "t_statistic": t_stat,
                "p_value": p_value,
            }
        )

    results_df = pd.DataFrame(results)

    # BH correction
    from scipy.stats import rankdata

    n = len(results_df)
    ranked_pvals = rankdata(results_df["p_value"])
    results_df["p_adjusted"] = results_df["p_value"] * n / ranked_pvals
    results_df["p_adjusted"] = results_df["p_adjusted"].clip(upper=1.0)

    results_df["significant"] = results_df["p_adjusted"] < significance_threshold
    results_df = results_df.sort_values("p_value")

    return results_df


dp_results = differential_protein_analysis(
    prot_df, pheno_df["phenotype"], CONFIG.significance_threshold
)

sig_proteins = dp_results[dp_results["significant"]]
print(
    f"Differentially abundant proteins (p_adj < {CONFIG.significance_threshold}): {len(sig_proteins)}"
)
print("\nTop significant proteins:")
print(
    dp_results.head(10)[
        ["protein", "mean_difference", "p_value", "p_adjusted", "significant"]
    ].to_string(index=False)
)

Differentially abundant proteins (p_adj < 0.05): 0

Top significant proteins:
 protein  mean_difference      p_value  p_adjusted  significant
    2625        -1.664453 1.386736e-36         NaN        False
     367        -1.611962 1.021536e-33         NaN        False
    2099        -1.601717 3.491822e-33         NaN        False
    8821        -1.367164 7.255242e-23         NaN        False
     898         1.355655 1.934573e-22         NaN        False
    2956         1.231785 3.044889e-18         NaN        False
     440         1.160046 4.174242e-16         NaN        False
    2305         1.154568 5.966333e-16         NaN        False
    5601        -1.144627 1.133335e-15         NaN        False
   57580        -1.128214 3.210924e-15         NaN        False


## 7. SNP-Phenotype Association

In [8]:
def snp_phenotype_association(
    snp_df: pd.DataFrame, pheno: pd.Series, significance_threshold: float = 0.05
) -> pd.DataFrame:
    """Chi-square test for SNP-phenotype association."""

    results = []

    for snp in snp_df.columns:
        # Create contingency table
        contingency = pd.crosstab(snp_df[snp], pheno)

        # Chi-square test
        if contingency.shape == (2, 2):
            chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
        else:
            # Skip if not a 2x2 table
            chi2, p_value = 0, 1.0

        # Calculate odds ratio
        if contingency.shape == (2, 2):
            a, b = contingency.iloc[0, :]
            c, d = contingency.iloc[1, :]
            odds_ratio = (a * d) / (b * c + 1e-10)
        else:
            odds_ratio = 1.0

        results.append(
            {
                "snp": snp,
                "chi2": chi2,
                "p_value": p_value,
                "odds_ratio": odds_ratio,
                "variant_freq_responder": snp_df.loc[pheno == "responder", snp].mean(),
                "variant_freq_non_responder": snp_df.loc[
                    pheno == "non_responder", snp
                ].mean(),
            }
        )

    results_df = pd.DataFrame(results)

    # BH correction
    from scipy.stats import rankdata

    n = len(results_df)
    ranked_pvals = rankdata(results_df["p_value"])
    results_df["p_adjusted"] = results_df["p_value"] * n / ranked_pvals
    results_df["p_adjusted"] = results_df["p_adjusted"].clip(upper=1.0)

    results_df["significant"] = results_df["p_adjusted"] < significance_threshold
    results_df = results_df.sort_values("p_value")

    return results_df


snp_assoc = snp_phenotype_association(
    snp_df, pheno_df["phenotype"], CONFIG.significance_threshold
)

sig_snps = snp_assoc[snp_assoc["significant"]]
print(
    f"Significant SNP-phenotype associations (p_adj < {CONFIG.significance_threshold}): {len(sig_snps)}"
)
print("\nTop SNP associations:")
print(
    snp_assoc.head(10)[
        ["snp", "odds_ratio", "p_value", "p_adjusted", "significant"]
    ].to_string(index=False)
)

Significant SNP-phenotype associations (p_adj < 0.05): 2

Top SNP associations:
    snp  odds_ratio      p_value   p_adjusted  significant
   TP53   19.206731 3.268556e-17 3.268556e-15         True
 PIK3CA    0.092986 6.172284e-06 3.086142e-04         True
  GATA3    0.000000 8.175648e-03 2.725216e-01        False
     TG    7.300000 8.829745e-03 2.207436e-01        False
   CDH1    0.106061 1.594375e-02 3.188751e-01        False
 MYO18B    5.450000 2.118101e-02 3.530168e-01        False
DYNC2H1    4.340000 4.233207e-02 6.047439e-01        False
  SPTA1    3.275510 6.143491e-02 7.679364e-01        False
    TTN    2.003571 7.336364e-02 8.151516e-01        False
  MUC5B    4.274510 8.557761e-02 8.557761e-01        False


## 8. Feature Importance from Random Forest

In [9]:
def get_rf_feature_importance(
    X: pd.DataFrame, y: pd.Series, seed: int, n_top: int = 20
) -> pd.DataFrame:
    """Extract feature importance from Random Forest model."""

    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Train Random Forest
    rf = RandomForestClassifier(n_estimators=100, random_state=seed)
    rf.fit(X_scaled, y)

    # Get feature importances
    importance_df = pd.DataFrame(
        {
            "feature": X.columns,
            "importance": rf.feature_importances_,
        }
    ).sort_values("importance", ascending=False)

    return importance_df.head(n_top)


rf_importance = get_rf_feature_importance(
    X_integrated, y, CONFIG.random_seed, CONFIG.top_features_to_analyze
)

print("Top important features (Random Forest):")
print(rf_importance.to_string(index=False))

Top important features (Random Forest):
feature  importance
  10551    0.071607
  79083    0.064215
  25803    0.063244
  23158    0.053871
   3169    0.050064
   2813    0.042207
   2099    0.039889
   7031    0.039633
   7033    0.037981
  10103    0.029829
  79875    0.028255
   2625    0.026548
 2625.1    0.022064
   3868    0.020993
 124220    0.019945
    771    0.017426
   3854    0.015064
   2001    0.013085
      9    0.012065
   2568    0.011692


## 9. Validate with Unit Tests

In [10]:
# Validation assertions
assert len(de_results) > 0, "Differential expression analysis must produce results"
assert "p_adjusted" in de_results.columns, "Must include multiple testing correction"

assert len(dp_results) > 0, "Differential protein analysis must produce results"

assert len(snp_assoc) > 0, "SNP association analysis must produce results"

assert len(rf_importance) > 0, "RF feature importance must be calculated"

print("[OK] All validation tests passed!")

[OK] All validation tests passed!


## 10. Export Results

In [11]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Export differential expression results
de_file = EXPORT_DIR / "task3_differential_expression.csv"
de_results.to_csv(de_file, index=False)

# Export differential protein results
dp_file = EXPORT_DIR / "task3_differential_proteins.csv"
dp_results.to_csv(dp_file, index=False)

# Export SNP associations
snp_file = EXPORT_DIR / "task3_snp_associations.csv"
snp_assoc.to_csv(snp_file, index=False)

# Export RF feature importance
importance_file = EXPORT_DIR / "task3_feature_importance.csv"
rf_importance.to_csv(importance_file, index=False)

# Export top variable features with annotations
top_genes_file = EXPORT_DIR / "task3_top_variable_genes.csv"
top_genes_annotated.to_csv(top_genes_file, index=False)

print(f"[OK] Differential expression saved to: {de_file.resolve()}")
print(f"[OK] Differential proteins saved to: {dp_file.resolve()}")
print(f"[OK] SNP associations saved to: {snp_file.resolve()}")
print(f"[OK] Feature importance saved to: {importance_file.resolve()}")
print(f"[OK] Top variable genes saved to: {top_genes_file.resolve()}")

[OK] Differential expression saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/10_integrative/assignments/artifacts/task3_differential_expression.csv
[OK] Differential proteins saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/10_integrative/assignments/artifacts/task3_differential_proteins.csv
[OK] SNP associations saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/10_integrative/assignments/artifacts/task3_snp_associations.csv
[OK] Feature importance saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/10_integrative/assignments/artifacts/task3_feature_importance.csv
[OK] Top variable genes saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/10_integrative/assignments/artifacts/task3_top_variable_genes.csv


In [12]:
# Summary
print("\n" + "=" * 60)
print("TASK 3 BIOLOGICAL INTERPRETATION SUMMARY")
print("=" * 60)

print(f"\nDifferential Expression:")
print(f"  Significant genes (p_adj < 0.05): {len(sig_genes)}")
if len(sig_genes) > 0:
    print(
        f"  Top gene: {de_results.iloc[0]['gene']} (p={de_results.iloc[0]['p_value']:.2e})"
    )

print(f"\nDifferential Proteins:")
print(f"  Significant proteins (p_adj < 0.05): {len(sig_proteins)}")
if len(sig_proteins) > 0:
    print(
        f"  Top protein: {dp_results.iloc[0]['protein']} (p={dp_results.iloc[0]['p_value']:.2e})"
    )

print(f"\nSNP Associations:")
print(f"  Significant SNPs (p_adj < 0.05): {len(sig_snps)}")

print(f"\nTop RF Features:")
for i, row in rf_importance.head(5).iterrows():
    print(f"  {row['feature']}: {row['importance']:.4f}")

print("=" * 60)


TASK 3 BIOLOGICAL INTERPRETATION SUMMARY

Differential Expression:
  Significant genes (p_adj < 0.05): 690
  Top gene: 3169 (p=3.38e-90)

Differential Proteins:
  Significant proteins (p_adj < 0.05): 0

SNP Associations:
  Significant SNPs (p_adj < 0.05): 2

Top RF Features:
  10551: 0.0716
  79083: 0.0642
  25803: 0.0632
  23158: 0.0539
  3169: 0.0501
